In [1]:
import pandas as pd
from PIL import Image
import os
from tqdm import tqdm

from transformers import AutoModel, AutoTokenizer
import torch

In [2]:
model_name = 'minicpm'
prompt_type = 'zeroshot'

In [3]:
dataset = 'type1'
image_type = 'combined'
device = "cuda:0"

In [4]:
qas = pd.read_json(f'../{dataset}/qa.json')
qas

,question_id,image_index,qa_id,qa_type,question,answer
0,0,5,qa_8,Original,Which year recorded the least daily hempseed p...,1990
1,1,5,qa_9,Original,Which of the following saw the higher daily he...,Europe
2,2,5,qa_1,LLM,What is the total hempseed production for the ...,The total hempseed production for the world in...
3,3,5,qa_3,LLM,What is the percentage increase in daily hemps...,The percentage increase in daily hempseed prod...
4,4,5,qa_5,LLM,What is the ratio of hempseed production in Eu...,The ratio is 0.236
...,...,...,...,...,...,...
2804,2804,900,qa_3,LLM,In which year was there the highest ratio of q...,2019
2805,2805,900,qa_11,Original,What is the maximum crime rate shown in the ch...,8000
2806,2806,900,qa_12,LLM,In which year did the most significant increas...,2019
2807,2807,900,qa_12,Original,In which year did the maximum crime rate occur?,2002


In [5]:
image_indices = qas['image_index'].values.astype(int)
questions = qas['question'].values
answers = qas['answer'].values

In [6]:
image_base_path = f'../{dataset}/{image_type}/'

In [7]:
all_images = os.listdir(image_base_path)
index_to_image = {}

if dataset == 'type1':
    prefix = 'multichart_'
    if image_type == 'original' or image_type == 'simple':
        sep = '_'
    else:
        sep = '.'

for index in set(image_indices):
    for image in all_images:
        if image.startswith(f'{prefix}{index}{sep}'):
            if index not in index_to_image:
                index_to_image[index] = []
            index_to_image[index].append(image)

index_to_image

{np.int64(5): ['multichart_5.png'],
 np.int64(6): ['multichart_6.png'],
 np.int64(7): ['multichart_7.png'],
 np.int64(8): ['multichart_8.png'],
 np.int64(9): ['multichart_9.png'],
 np.int64(52): ['multichart_52.png'],
 np.int64(53): ['multichart_53.png'],
 np.int64(54): ['multichart_54.png'],
 np.int64(55): ['multichart_55.png'],
 np.int64(56): ['multichart_56.png'],
 np.int64(57): ['multichart_57.png'],
 np.int64(58): ['multichart_58.png'],
 np.int64(59): ['multichart_59.png'],
 np.int64(61): ['multichart_61.png'],
 np.int64(62): ['multichart_62.png'],
 np.int64(63): ['multichart_63.png'],
 np.int64(64): ['multichart_64.png'],
 np.int64(65): ['multichart_65.png'],
 np.int64(66): ['multichart_66.png'],
 np.int64(67): ['multichart_67.png'],
 np.int64(68): ['multichart_68.png'],
 np.int64(69): ['multichart_69.png'],
 np.int64(70): ['multichart_70.png'],
 np.int64(71): ['multichart_71.png'],
 np.int64(72): ['multichart_72.png'],
 np.int64(73): ['multichart_73.png'],
 np.int64(74): ['multi

In [8]:
message_template = [
    {
        "role": "user",
        "content": []
    }
]

In [9]:
def get_message(image_index, prompt, question):
    images = []
    for image in index_to_image[image_index]:
        image_path = f'{image_base_path}{image}'
        images.append(Image.open(image_path).convert('RGB'))
    message = message_template.copy()
    message[0]['content'] = images
    message[0]['content'].append(prompt.format(question))
    return message

In [10]:
model = AutoModel.from_pretrained('openbmb/MiniCPM-V-2_6',                                                    
    trust_remote_code=True,
    attn_implementation='flash_attention_2', 
    torch_dtype=torch.bfloat16,
    device_map = device,
    cache_dir='/media/vivek/drive/multichartqa/models_cache') # sdpa or flash_attention_2, no eager

model = model.eval()
tokenizer = AutoTokenizer.from_pretrained('openbmb/MiniCPM-V-2_6', trust_remote_code=True)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [11]:
model_responses = []

In [12]:
def run_model():
    for i, question in enumerate(tqdm(questions)):
        image_index = image_indices[i]
        prompt = """Your task is the answer the question based on the given image. Your final answer to the question should strictly be in the format - "Final Answer:" <final_answer>.\n\nQuestion: {}""" 
        messages = get_message(image_index, prompt, question)
        answer = model.chat(image=None, msgs=messages, tokenizer=tokenizer)
        model_responses.append({'question_id': i, 'question': question, 'gold': answers[i], 'response': answer.strip()})
        # print(f'Question: {question}\nAnswer: {answers[i]}\nResponse: {answer.strip()}\n\n')

In [13]:
run_model()

  0%|          | 0/2809 [00:00<?, ?it/s]/home/vivek/anaconda3/envs/multichartqa/lib/python3.10/site-packages/transformers/models/auto/image_processing_auto.py:518: FutureWarning: The image_processor_class argument is deprecated and will be removed in v4.42. Please use `slow_image_processor_class`, or `fast_image_processor_class` instead
  warnings.warn(
Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)
 70%|███████   | 1977/2809 [3:33:32<1:19:28,  5.73s/it]

In [14]:
# saving the model responses
os.makedirs(f'../model_responses/{dataset}', exist_ok=True)

model_responses_df = pd.DataFrame(model_responses)
model_responses_df.to_json(f'../model_responses/{dataset}/{model_name}_{image_type}_{prompt_type}.json', orient='records')